# npm 패키지 보안 위험도 평가 시스템 v3
## Hybrid Rule-Based + Gemini Explanation

**단국대학교 소프트웨어학과 32230120 강하늘**

---

### 프로젝트 개요

이 노트북은 npm 패키지의 보안 위험도를 **0~100점** 범위에서 평가하는 하이브리드 시스템을 구현합니다.

#### 핵심 문제의식
기존 v1/v2 모델(RandomForest + XGBoost)은 `maintainers_count` 피처에 43.8% 중요도가 쏠려
결과가 0 또는 100으로만 이진화되는 문제가 있었습니다.
이를 해결하기 위해 **규칙 기반 가중치 점수 산정**으로 방식을 전환합니다.

#### 아키텍처
```
[1단계] 규칙 기반 위험도 점수 산정 (0~100)
  - 타이포스쿼팅 위험도 (최대 30점)
  - 패키지 미존재 (최대 15점)
  - 설치 스크립트 (최대 20점)
  - 메타데이터 신호 (최대 15점)
  - 다운로드/관리자 신호 (최대 20점)
  → 합계 최대 100점

[2단계] Gemini API 자연어 설명 생성
  - 점수 근거를 한국어로 설명
  - 41~60점 구간: 추가 검토 필요 안내

[3단계] 평가 및 시각화
  - 점수 분포 그래프
  - 피처 기여도 히트맵
  - Precision/Recall/F1
```


## Step 0. 환경 설정

In [ ]:
!pip install -q google-generativeai requests pandas matplotlib seaborn tabulate

## Step 1. 라이브러리 임포트 & API 설정

In [ ]:
import requests
import time
import json
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import google.generativeai as genai
import warnings
warnings.filterwarnings('ignore')

# ─── Gemini API 설정 ───────────────────────────────────────────────────────
# Colab Secrets 사용 방법:
#   1. 왼쪽 열쇠(🔑) 아이콘 클릭
#   2. "+ Add new secret" → Name: GEMINI_API_KEY, Value: (본인 API 키 입력)
#   3. "Notebook access" 토글 ON 후 저장
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    import os
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY가 설정되지 않았습니다. Colab Secrets에 키를 추가하세요.")

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('gemini-2.0-flash')

print("✅ Gemini API 초기화 완료")
print(f"   모델: gemini-2.0-flash")

## Step 2. 유명 패키지 목록 (Typosquatting 기준)

In [ ]:
# npm 주간 다운로드 상위 50개 패키지 기준
# (Slopsquatting 공격은 이 목록을 위장 대상으로 활용)
POPULAR_PACKAGES = [
    "react", "lodash", "express", "axios", "chalk",
    "moment", "webpack", "babel", "typescript", "eslint",
    "prettier", "jest", "mocha", "mongoose", "sequelize",
    "async", "bluebird", "request", "got", "node-fetch",
    "commander", "yargs", "minimist", "glob", "rimraf",
    "mkdirp", "through", "mime", "uuid", "semver",
    "debug", "ms", "qs", "bytes", "body-parser",
    "cors", "dotenv", "jsonwebtoken", "bcrypt", "bcryptjs",
    "yaml", "cheerio", "puppeteer", "playwright", "socket.io",
    "nodemailer", "multer", "sharp", "jimp", "pdfkit",
]

print(f"📦 유명 패키지 목록: {len(POPULAR_PACKAGES)}개")

## Step 3. npm Registry API 함수

In [ ]:
REGISTRY_URL = "https://registry.npmjs.org"
DOWNLOADS_URL = "https://api.npmjs.org/downloads/point/last-week"

def fetch_package_info(name: str) -> dict:
    """npm 레지스트리에서 패키지 메타데이터 조회"""
    try:
        resp = requests.get(f"{REGISTRY_URL}/{name}", timeout=10)
        if resp.status_code == 404:
            return {"exists": False}
        if resp.status_code != 200:
            return {"exists": False, "error": resp.status_code}
        data = resp.json()
        latest_version = data.get("dist-tags", {}).get("latest", "")
        latest_data = data.get("versions", {}).get(latest_version, {})
        scripts = latest_data.get("scripts", {})
        return {
            "exists": True,
            "description": data.get("description", ""),
            "maintainers": data.get("maintainers", []),
            "maintainers_count": len(data.get("maintainers", [])),
            "keywords": data.get("keywords", []),
            "repository": data.get("repository", {}),
            "scripts": scripts,
            "has_install_script": any(
                k in scripts for k in ["preinstall", "install", "postinstall"]
            ),
            "latest_version": latest_version,
            "dependencies_count": len(latest_data.get("dependencies", {})),
        }
    except Exception as e:
        return {"exists": False, "error": str(e)}

def fetch_weekly_downloads(name: str) -> int:
    """npm 주간 다운로드 수 조회"""
    try:
        resp = requests.get(f"{DOWNLOADS_URL}/{name}", timeout=10)
        if resp.status_code == 200:
            return resp.json().get("downloads", 0)
    except:
        pass
    return 0

def levenshtein(a: str, b: str) -> int:
    """편집 거리 계산 (O(min(m,n)) space)"""
    m, n = len(a), len(b)
    if m < n:
        a, b, m, n = b, a, n, m
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            temp = dp[j]
            dp[j] = prev if a[i-1] == b[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev = temp
    return dp[n]

def min_edit_distance(name: str, popular: list) -> tuple:
    """유명 패키지 목록 대비 최소 편집 거리 계산"""
    if name in popular:
        return 0, name  # 정확히 일치 → 안전
    distances = [(levenshtein(name, p), p) for p in popular]
    min_dist, closest = min(distances, key=lambda x: x[0])
    return min_dist, closest

print("✅ npm API 함수 준비 완료")

## Step 4. 가설 설정

점수 산정 전, 검증할 가설을 먼저 명시합니다.

| 가설 번호 | 내용 | 검증 방법 |
|---|---|---|
| H1 | 편집 거리 1인 타이포스쿼팅 패키지는 HIGH 이상(60+)이어야 한다 | 해당 패키지들의 평균 점수 확인 |
| H2 | npm에 존재하지 않는(404) 패키지는 존재하는 패키지보다 점수가 15점 높아야 한다 | 존재 여부별 평균 점수 비교 |
| H3 | 설치 스크립트가 있는 패키지는 없는 패키지보다 평균 20점 높아야 한다 | 설치 스크립트 여부별 평균 비교 |
| H4 | 유명 패키지(정확 일치)는 20점 이하여야 한다 | 유명 패키지 카테고리 점수 확인 |
| H5 | MEDIUM(41~60) 구간에 패키지가 적절히 분포해야 한다 | 구간별 패키지 수 분포 |


## Step 5. 위험도 점수 산정 함수 (규칙 기반)

### 점수 구성 (합계 최대 100점)

| 항목 | 최대 점수 | 기준 |
|---|---:|---|
| 타이포스쿼팅 위험도 | 30 | edit_dist 1→30, 2→20, 3→10, 4+→0 |
| 패키지 미존재 (404) | 15 | npm에 없으면 +15 |
| 설치 스크립트 존재 | 20 | preinstall/install/postinstall |
| 설명 없음 | 8 | description 필드 비어있음 |
| 저장소 링크 없음 | 5 | repository 필드 없음 |
| 키워드 없음 | 2 | keywords 비어있음 |
| 낮은 다운로드 수 | 10 | 0→10, <100→7, <1K→4, <10K→1 |
| 관리자 수 부족 | 10 | 0명→10, 1명→5, 2명→2, 3+→0 |
| **합계 최대** | **100** | |


In [ ]:
def compute_risk_score(features: dict) -> tuple:
    """
    규칙 기반 위험도 점수 산정
    Returns: (score: int, components: dict)
    """
    score = 0
    components = {}

    # ─── A. 패키지 미존재 (0~15) ─────────────────────────────────────────
    if not features["exists"]:
        components["패키지 미존재(404)"] = 15
        score += 15
    else:
        components["패키지 미존재(404)"] = 0

    # ─── B. 타이포스쿼팅 위험도 (0~30) ──────────────────────────────────
    min_dist = features["min_edit_dist"]
    if min_dist == 0:
        typo_score = 0  # 유명 패키지 본인 → 안전
    elif min_dist == 1:
        typo_score = 30
    elif min_dist == 2:
        typo_score = 20
    elif min_dist == 3:
        typo_score = 10
    else:
        typo_score = 0
    components["타이포스쿼팅 위험도"] = typo_score
    score += typo_score

    # ─── C. 설치 스크립트 (0~20) ─────────────────────────────────────────
    if features.get("has_install_script", False):
        components["설치 스크립트 존재"] = 20
        score += 20
    else:
        components["설치 스크립트 존재"] = 0

    # ─── D. 설명 없음 (0~8) ──────────────────────────────────────────────
    if not features.get("description", ""):
        components["설명 없음"] = 8
        score += 8
    else:
        components["설명 없음"] = 0

    # ─── E. 저장소 링크 없음 (0~5) ───────────────────────────────────────
    repo = features.get("repository", {})
    has_repo = bool(repo and (isinstance(repo, dict) and repo.get("url")) or isinstance(repo, str))
    if not has_repo:
        components["저장소 링크 없음"] = 5
        score += 5
    else:
        components["저장소 링크 없음"] = 0

    # ─── F. 키워드 없음 (0~2) ────────────────────────────────────────────
    if not features.get("keywords", []):
        components["키워드 없음"] = 2
        score += 2
    else:
        components["키워드 없음"] = 0

    # ─── G. 낮은 다운로드 수 (0~10) ──────────────────────────────────────
    dl = features.get("weekly_downloads", 0)
    if dl == 0:
        dl_score = 10
    elif dl < 100:
        dl_score = 7
    elif dl < 1_000:
        dl_score = 4
    elif dl < 10_000:
        dl_score = 1
    else:
        dl_score = 0
    components["낮은 다운로드 수"] = dl_score
    score += dl_score

    # ─── H. 관리자 수 부족 (0~10) ────────────────────────────────────────
    m = features.get("maintainers_count", 0)
    if m == 0:
        m_score = 10
    elif m == 1:
        m_score = 5
    elif m == 2:
        m_score = 2
    else:
        m_score = 0
    components["관리자 수 부족"] = m_score
    score += m_score

    return min(score, 100), components


def get_grade(score: int) -> dict:
    """점수 → 등급 변환"""
    if score <= 20:
        return {"grade": "LOW",      "label": "낮은 위험",      "color": "#22c55e", "threshold": "safe"}
    elif score <= 40:
        return {"grade": "LOW-MED",  "label": "비교적 낮은 위험", "color": "#84cc16", "threshold": "low"}
    elif score <= 60:
        return {"grade": "MEDIUM",   "label": "중간 위험",      "color": "#f59e0b", "threshold": "medium"}
    elif score <= 80:
        return {"grade": "HIGH",     "label": "높은 위험",      "color": "#f97316", "threshold": "high"}
    else:
        return {"grade": "CRITICAL", "label": "매우 높은 위험",  "color": "#ef4444", "threshold": "critical"}


print("✅ 점수 산정 함수 준비 완료")
print("   최대 가능 점수: 15+30+20+8+5+2+10+10 = 100점")

## Step 6. Gemini 설명 생성 함수

In [ ]:
def generate_explanation(package_name: str, score: int, grade_info: dict,
                         features: dict, components: dict) -> str:
    """
    Gemini API를 사용하여 위험도 점수에 대한 자연어 설명 생성
    NOTE: Gemini는 점수 산정 자체를 하지 않습니다.
          규칙 기반으로 산정된 점수에 대한 설명만 생성합니다.
    """
    grade = grade_info["grade"]
    label = grade_info["label"]

    # 점수 구성 요소 텍스트 생성
    active_components = [(k, v) for k, v in components.items() if v > 0]
    comp_text = "\n".join([f"  - {k}: +{v}점" for k, v in active_components])
    if not comp_text:
        comp_text = "  - 위험 신호 없음 (0점)"

    # 패키지 특성 텍스트
    dist = features.get('min_edit_dist', 99)
    closest = features.get('closest_package', '없음')
    dl = features.get('weekly_downloads', 0)
    exists = features.get('exists', False)
    has_script = features.get('has_install_script', False)
    maintainers = features.get('maintainers_count', 0)

    # 구간별 추가 안내 문구
    threshold = grade_info["threshold"]
    if threshold == "safe":
        guidance = "현재 분석 기준으로는 위험도가 낮아 보입니다."
    elif threshold == "low":
        guidance = "큰 위험 신호는 없지만 일부 항목은 확인이 필요합니다."
    elif threshold == "medium":
        guidance = "⚠️ 위험도가 애매한 구간입니다. 추가 코드 검토 또는 의존성 확인을 권장합니다."
    elif threshold == "high":
        guidance = "🚨 위험 신호가 다수 확인됩니다. 사용 전 신중한 검토가 필요합니다."
    else:
        guidance = "🚨🚨 매우 높은 위험 가능성이 있습니다. 사용을 보류하고 정밀 분석이 필요합니다."

    prompt = f"""당신은 npm 패키지 보안 전문가입니다.
아래 패키지에 대해 규칙 기반 점수 산정 결과를 바탕으로 간결하고 전문적인 보안 설명을 작성해주세요.

패키지명: {package_name}
위험도 점수: {score}/100
위험도 등급: {grade} ({label})

점수 구성 요소:
{comp_text}

패키지 특성:
- npm 존재 여부: {'존재함' if exists else '존재하지 않음 (404)'}
- 유명 패키지와의 편집 거리: {dist} (가장 가까운: {closest})
- 주간 다운로드: {dl:,}회
- 관리자 수: {maintainers}명
- 설치 스크립트: {'있음' if has_script else '없음'}

다음 형식으로 한국어로 답변해주세요 (각 항목 1~2문장, 총 5줄 이내):
【주요 판단 근거】 ...
【주의 요소】 ...
【최종 안내】 {guidance}"""

    try:
        response = gemini_model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f"(설명 생성 실패: {e})\n【최종 안내】 {guidance}"


print("✅ Gemini 설명 생성 함수 준비 완료")
print("   Gemini 역할: 점수 산정 X, 설명 생성 O")

## Step 7. 데이터셋 구성

20개 패키지를 선정하여 위험도 분포를 검증합니다.

| 카테고리 | 패키지 수 | 설명 |
|---|---:|---|
| SAFE | 5 | 널리 사용되는 정상 패키지 |
| LOW | 3 | 소규모이지만 정상 패키지 |
| MEDIUM | 4 | 설치 스크립트 있는 정상 패키지 |
| HIGH | 4 | 타이포스쿼팅 의심 패키지 |
| CRITICAL | 4 | npm 미존재 + 타이포스쿼팅 복합 |


In [ ]:
# true_label: 우리가 기대하는 위험 등급 (평가 지표 계산에 사용)
PACKAGES = [
    # ─── SAFE: 유명 정상 패키지 ──────────────────────────────────────────
    {"name": "lodash",     "true_label": "SAFE",     "note": "JS 유틸리티 라이브러리 1위"},
    {"name": "express",    "true_label": "SAFE",     "note": "Node.js 웹 프레임워크 1위"},
    {"name": "axios",      "true_label": "SAFE",     "note": "HTTP 클라이언트 1위"},
    {"name": "react",      "true_label": "SAFE",     "note": "UI 라이브러리 1위"},
    {"name": "chalk",      "true_label": "SAFE",     "note": "터미널 색상 라이브러리"},

    # ─── LOW: 소규모 정상 패키지 ─────────────────────────────────────────
    {"name": "ms",         "true_label": "LOW",      "note": "시간 단위 변환 유틸리티"},
    {"name": "mime",       "true_label": "LOW",      "note": "MIME 타입 감지"},
    {"name": "uuid",       "true_label": "LOW",      "note": "UUID 생성 라이브러리"},

    # ─── MEDIUM: 설치 스크립트 있는 정상 패키지 ──────────────────────────
    {"name": "bcrypt",         "true_label": "MEDIUM",   "note": "암호화, C++ 네이티브 빌드"},
    {"name": "node-pre-gyp",   "true_label": "MEDIUM",   "note": "네이티브 애드온 빌드 도구"},
    {"name": "node-sass",      "true_label": "MEDIUM",   "note": "deprecated, 네이티브 빌드"},
    {"name": "puppeteer",      "true_label": "MEDIUM",   "note": "Chromium 자동 다운로드"},

    # ─── HIGH: 타이포스쿼팅 의심 패키지 ──────────────────────────────────
    {"name": "loadash",    "true_label": "HIGH",     "note": "lodash 타이포스쿼팅"},
    {"name": "expresss",   "true_label": "HIGH",     "note": "express 타이포스쿼팅"},
    {"name": "chak",       "true_label": "HIGH",     "note": "chalk 타이포스쿼팅"},
    {"name": "axois",      "true_label": "HIGH",     "note": "axios 타이포스쿼팅"},

    # ─── CRITICAL: npm 미존재 + 타이포스쿼팅 복합 ────────────────────────
    {"name": "axio",                  "true_label": "CRITICAL", "note": "axios 편집거리1, 404"},
    {"name": "yaml-stream-parser",    "true_label": "CRITICAL", "note": "실제 데모 패키지, 404"},
    {"name": "react-hook-form-v2",    "true_label": "CRITICAL", "note": "버전 위장, 404"},
    {"name": "lodash-utils-extra",    "true_label": "CRITICAL", "note": "lodash 위장, 404"},
]

# 라벨 → 이진 변환 (평가 지표용)
BINARY_LABEL = {
    "SAFE": 0, "LOW": 0, "MEDIUM": 1, "HIGH": 1, "CRITICAL": 1
}

print(f"📊 데이터셋: 총 {len(PACKAGES)}개 패키지")
for cat in ["SAFE", "LOW", "MEDIUM", "HIGH", "CRITICAL"]:
    count = sum(1 for p in PACKAGES if p["true_label"] == cat)
    print(f"   {cat}: {count}개")

## Step 8. 데이터 수집 (npm API 호출)

각 패키지의 메타데이터와 다운로드 수를 npm Registry에서 실시간으로 수집합니다.

In [ ]:
raw_data = []

for pkg in PACKAGES:
    name = pkg["name"]
    print(f"  📦 수집 중: {name}", end=" ... ")

    info = fetch_package_info(name)
    downloads = fetch_weekly_downloads(name) if info.get("exists") else 0
    min_dist, closest = min_edit_distance(name, POPULAR_PACKAGES)

    entry = {
        **pkg,
        "exists": info.get("exists", False),
        "description": info.get("description", ""),
        "maintainers_count": info.get("maintainers_count", 0),
        "keywords": info.get("keywords", []),
        "repository": info.get("repository", {}),
        "has_install_script": info.get("has_install_script", False),
        "weekly_downloads": downloads,
        "min_edit_dist": min_dist,
        "closest_package": closest,
        "dependencies_count": info.get("dependencies_count", 0),
    }
    raw_data.append(entry)

    status = "✅ exists" if info.get("exists") else "❌ 404"
    print(f"{status} | dist={min_dist}→{closest} | dl={downloads:,}")
    time.sleep(0.3)  # API rate limit 방지

print(f"\n✅ 데이터 수집 완료: {len(raw_data)}개")

## Step 9. 위험도 점수 산정

In [ ]:
scored_data = []

for entry in raw_data:
    score, components = compute_risk_score(entry)
    grade_info = get_grade(score)
    scored_data.append({
        **entry,
        "score": score,
        "grade": grade_info["grade"],
        "grade_label": grade_info["label"],
        "color": grade_info["color"],
        "threshold": grade_info["threshold"],
        "components": components,
    })

# 결과 DataFrame 생성
df = pd.DataFrame(scored_data)

# 점수 분포 출력
print("\n📊 카테고리별 평균 점수:")
for cat in ["SAFE", "LOW", "MEDIUM", "HIGH", "CRITICAL"]:
    subset = df[df['true_label'] == cat]['score']
    if len(subset) > 0:
        print(f"   {cat:10s}: 평균 {subset.mean():.1f}점 (min={subset.min()}, max={subset.max()})")

print(f"\n   전체 평균: {df['score'].mean():.1f}점")
print(f"   최소: {df['score'].min()}점 / 최대: {df['score'].max()}점")
print(f"   표준편차: {df['score'].std():.1f}점")

## Step 10. Gemini 설명 생성

규칙 기반으로 산정된 점수를 바탕으로 Gemini가 자연어 설명을 생성합니다.

In [ ]:
print("🤖 Gemini 설명 생성 중...\n")

explanations = []
for entry in scored_data:
    name = entry["name"]
    grade_info = {"grade": entry["grade"], "label": entry["grade_label"], "threshold": entry["threshold"]}
    print(f"  📝 {name} (점수: {entry['score']}, {entry['grade']})")

    explanation = generate_explanation(
        name, entry["score"], grade_info, entry, entry["components"]
    )
    explanations.append(explanation)
    time.sleep(1.5)  # Gemini API rate limit

df["explanation"] = explanations
print("\n✅ 설명 생성 완료")

## Step 11. 결과 출력

In [ ]:
print("=" * 80)
print("npm 패키지 보안 위험도 평가 결과")
print("=" * 80)

for _, row in df.sort_values("score", ascending=False).iterrows():
    score_bar = "█" * (row["score"] // 5) + "░" * (20 - row["score"] // 5)
    print(f"\n{'─'*70}")
    print(f"패키지명   : {row['name']}")
    print(f"위험도 점수: {row['score']:3d}/100  [{score_bar}]")
    print(f"위험도 등급: {row['grade']} ({row['grade_label']})")
    print(f"npm 존재   : {'✅ 있음' if row['exists'] else '❌ 404'}")
    print(f"다운로드   : {row['weekly_downloads']:,}회/주")
    print(f"가장 유사  : {row['closest_package']} (편집거리={row['min_edit_dist']})")

    # 점수 구성 요소
    active = [(k, v) for k, v in row["components"].items() if v > 0]
    if active:
        print(f"점수 구성  : " + " | ".join([f"{k}(+{v})" for k, v in active]))
    else:
        print(f"점수 구성  : 위험 신호 없음")

    print(f"Gemini 설명:")
    for line in row["explanation"].split("\n"):
        if line.strip():
            print(f"  {line}")

print(f"\n{'─'*70}")

## Step 12. 시각화

### 12-1. 패키지별 위험도 점수 막대 그래프

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

df_sorted = df.sort_values("score", ascending=True)
colors = df_sorted["color"].tolist()

bars = ax.barh(df_sorted["name"], df_sorted["score"], color=colors, edgecolor="white", linewidth=0.5)

# 점수 레이블
for bar, score in zip(bars, df_sorted["score"]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{score}", va="center", fontsize=10, fontweight="bold")

# 구간 배경
ax.axvspan(0,  20, alpha=0.05, color="#22c55e")
ax.axvspan(20, 40, alpha=0.05, color="#84cc16")
ax.axvspan(40, 60, alpha=0.08, color="#f59e0b")
ax.axvspan(60, 80, alpha=0.08, color="#f97316")
ax.axvspan(80, 100, alpha=0.08, color="#ef4444")

# 구간 경계선
for x, label in [(20, "LOW"), (40, "LOW-MED"), (60, "MEDIUM"), (80, "HIGH"), (100, "CRITICAL")]:
    ax.axvline(x=x, color="gray", linestyle="--", alpha=0.4, linewidth=1)

# 범례
legend_items = [
    mpatches.Patch(color="#22c55e", label="LOW (0-20)"),
    mpatches.Patch(color="#84cc16", label="LOW-MED (21-40)"),
    mpatches.Patch(color="#f59e0b", label="MEDIUM (41-60)"),
    mpatches.Patch(color="#f97316", label="HIGH (61-80)"),
    mpatches.Patch(color="#ef4444", label="CRITICAL (81-100)"),
]
ax.legend(handles=legend_items, loc="lower right", fontsize=9)

ax.set_xlabel("위험도 점수 (0~100)", fontsize=12)
ax.set_title("npm 패키지별 보안 위험도 점수", fontsize=14, fontweight="bold", pad=15)
ax.set_xlim(0, 108)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("risk_scores_by_package.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 저장: risk_scores_by_package.png")

### 12-2. 위험도 구간별 패키지 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 히스토그램
ax = axes[0]
bins = [0, 20, 40, 60, 80, 100]
bin_labels = ["LOW\n(0-20)", "LOW-MED\n(21-40)", "MEDIUM\n(41-60)", "HIGH\n(61-80)", "CRITICAL\n(81-100)"]
bin_colors = ["#22c55e", "#84cc16", "#f59e0b", "#f97316", "#ef4444"]

counts, _ = np.histogram(df["score"], bins=bins)
bars = ax.bar(range(5), counts, color=bin_colors, edgecolor="white", linewidth=1.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(count), ha="center", fontweight="bold", fontsize=12)

ax.set_xticks(range(5))
ax.set_xticklabels(bin_labels, fontsize=9)
ax.set_ylabel("패키지 수", fontsize=11)
ax.set_title("위험도 구간별 패키지 분포", fontsize=13, fontweight="bold")
ax.set_ylim(0, max(counts) + 1.5)
ax.grid(axis="y", alpha=0.3)

# 점수 히스토그램 (연속)
ax = axes[1]
ax.hist(df["score"], bins=20, color="#6366f1", edgecolor="white", alpha=0.8)
ax.axvline(df["score"].mean(), color="#ef4444", linestyle="--", linewidth=2,
           label=f"평균 {df['score'].mean():.1f}점")
ax.axvline(df["score"].median(), color="#f59e0b", linestyle="-.", linewidth=2,
           label=f"중앙값 {df['score'].median():.1f}점")
ax.set_xlabel("위험도 점수", fontsize=11)
ax.set_ylabel("빈도", fontsize=11)
ax.set_title("점수 분포 히스토그램", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("risk_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 저장: risk_distribution.png")

### 12-3. 위험 요인별 기여도 히트맵

In [ ]:
# 점수 구성 요소를 DataFrame으로 변환
component_keys = [
    "패키지 미존재(404)", "타이포스쿼팅 위험도", "설치 스크립트 존재",
    "설명 없음", "저장소 링크 없음", "키워드 없음",
    "낮은 다운로드 수", "관리자 수 부족"
]

component_df = pd.DataFrame([
    {k: row["components"].get(k, 0) for k in component_keys}
    for _, row in df.iterrows()
], index=df["name"])

# 패키지를 점수 순으로 정렬
sort_idx = df.sort_values("score", ascending=False)["name"].values
component_df = component_df.loc[sort_idx]

fig, ax = plt.subplots(figsize=(12, 8))

# 최대값으로 정규화 (시각적 강조)
max_vals = pd.Series({
    "패키지 미존재(404)": 15, "타이포스쿼팅 위험도": 30, "설치 스크립트 존재": 20,
    "설명 없음": 8, "저장소 링크 없음": 5, "키워드 없음": 2,
    "낮은 다운로드 수": 10, "관리자 수 부족": 10
})
normalized_df = component_df.div(max_vals) * 100

sns.heatmap(
    normalized_df,
    annot=component_df.values,
    fmt="g",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "기여도 (%)"},
    vmin=0,
    vmax=100,
)

ax.set_title("패키지별 위험 요인 기여도 히트맵 (숫자=실제 점수)", fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("위험 요인", fontsize=11)
ax.set_ylabel("패키지명", fontsize=11)
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig("risk_factor_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 저장: risk_factor_heatmap.png")

### 12-4. 예측 점수 vs 실제 라벨 비교

In [ ]:
label_order = ["SAFE", "LOW", "MEDIUM", "HIGH", "CRITICAL"]
label_colors = {"SAFE": "#22c55e", "LOW": "#84cc16", "MEDIUM": "#f59e0b", "HIGH": "#f97316", "CRITICAL": "#ef4444"}

fig, ax = plt.subplots(figsize=(12, 5))

for label in label_order:
    subset = df[df["true_label"] == label]
    ax.scatter(
        subset["name"], subset["score"],
        color=label_colors[label], s=120, zorder=5,
        label=f"{label} (n={len(subset)})"
    )

# 경계선
for y, label in [(20, "LOW"), (40, "LOW-MED"), (60, "MEDIUM"), (80, "HIGH")]:
    ax.axhline(y=y, color="gray", linestyle="--", alpha=0.5, linewidth=1)
    ax.text(len(df)-0.5, y+1, label, ha="right", fontsize=8, color="gray")

ax.set_ylim(-5, 108)
ax.set_ylabel("위험도 점수", fontsize=11)
ax.set_title("실제 라벨 vs 예측 점수 비교", fontsize=13, fontweight="bold")
ax.legend(loc="upper left", fontsize=9)
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("label_vs_score.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 저장: label_vs_score.png")

## Step 13. 평가 지표 계산

위험도 평가 모델의 성능을 정량적으로 측정합니다.

In [ ]:
# 이진 분류 기준: 점수 >= 40 → 위험(1), < 40 → 안전(0)
THRESHOLD = 40

y_true = [BINARY_LABEL[row["true_label"]] for _, row in df.iterrows()]
y_pred = [1 if row["score"] >= THRESHOLD else 0 for _, row in df.iterrows()]

precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Negative Rate

# 카테고리별 평균 점수
cat_means = df.groupby("true_label")["score"].agg(["mean", "std", "min", "max"])

print("=" * 60)
print(f"평가 지표 (임계값: {THRESHOLD}점)")
print("=" * 60)
print(f"  Precision (정밀도) : {precision:.3f}  [{tp}/{tp+fp}]")
print(f"  Recall    (재현율) : {recall:.3f}  [{tp}/{tp+fn}]")
print(f"  F1-Score           : {f1:.3f}")
print(f"  False Positive Rate: {fpr:.3f}  (정상→위험 오분류 {fp}개)")
print(f"  False Negative Rate: {fnr:.3f}  (위험→정상 미탐지 {fn}개)")
print()
print("혼동 행렬:")
print(f"  실제 안전→예측 안전(TN): {tn}  |  실제 안전→예측 위험(FP): {fp}")
print(f"  실제 위험→예측 안전(FN): {fn}  |  실제 위험→예측 위험(TP): {tp}")
print()
print("카테고리별 평균 점수:")
for label in ["SAFE", "LOW", "MEDIUM", "HIGH", "CRITICAL"]:
    if label in cat_means.index:
        row = cat_means.loc[label]
        print(f"  {label:10s}: 평균 {row['mean']:5.1f}점 ± {row['std']:4.1f} (min={row['min']}, max={row['max']})")

## Step 14. 가설 검증

In [ ]:
print("=" * 70)
print("가설 검증 결과")
print("=" * 70)

# H1: 편집 거리 1인 패키지 → 60점 이상
h1_packages = df[df["min_edit_dist"] == 1]
h1_mean = h1_packages["score"].mean() if len(h1_packages) > 0 else 0
h1_pass = h1_mean >= 60
print(f"\n[H1] 편집거리 1 패키지 → HIGH(60+) 예상")
print(f"     대상 패키지: {list(h1_packages['name'])}")
print(f"     평균 점수: {h1_mean:.1f}점")
print(f"     결과: {'✅ 검증됨' if h1_pass else '❌ 미검증'} (기준: 60점 이상)")

# H2: 404 패키지 vs 존재 패키지 점수 차이 >= 15
not_exists = df[~df["exists"]]["score"].mean() if len(df[~df["exists"]]) > 0 else 0
exists_pkg = df[df["exists"]]["score"].mean() if len(df[df["exists"]]) > 0 else 0
h2_diff = not_exists - exists_pkg
h2_pass = h2_diff >= 15
print(f"\n[H2] 404 패키지가 존재 패키지보다 15점 이상 높아야 함")
print(f"     404 패키지 평균: {not_exists:.1f}점 ({len(df[~df['exists']])}개)")
print(f"     존재 패키지 평균: {exists_pkg:.1f}점 ({len(df[df['exists']])}개)")
print(f"     차이: {h2_diff:.1f}점")
print(f"     결과: {'✅ 검증됨' if h2_pass else '❌ 미검증'} (기준: 15점 이상 차이)")

# H3: 설치 스크립트 있는 패키지 vs 없는 패키지 점수 차이 >= 20
with_script = df[df["has_install_script"]]["score"].mean() if len(df[df["has_install_script"]]) > 0 else 0
no_script   = df[~df["has_install_script"]]["score"].mean() if len(df[~df["has_install_script"]]) > 0 else 0
h3_diff = with_script - no_script
h3_pass = h3_diff >= 20
print(f"\n[H3] 설치 스크립트 있는 패키지가 20점 이상 높아야 함")
print(f"     설치 스크립트 있음: {with_script:.1f}점 ({len(df[df['has_install_script']])}개)")
print(f"     설치 스크립트 없음: {no_script:.1f}점")
print(f"     차이: {h3_diff:.1f}점")
print(f"     결과: {'✅ 검증됨' if h3_pass else '❌ 미검증'} (기준: 20점 이상 차이)")

# H4: 유명 패키지 (정확 일치) → 20점 이하
safe_pkgs = df[df["true_label"] == "SAFE"]
h4_max = safe_pkgs["score"].max() if len(safe_pkgs) > 0 else 999
h4_pass = h4_max <= 20
print(f"\n[H4] 유명 정상 패키지 → 20점 이하")
print(f"     최고 점수: {h4_max}점")
print(f"     결과: {'✅ 검증됨' if h4_pass else '❌ 미검증'} (기준: 20점 이하)")

# H5: MEDIUM 구간(41-60)에 패키지 분포
medium_count = len(df[(df["score"] >= 41) & (df["score"] <= 60)])
h5_pass = medium_count >= 2
print(f"\n[H5] MEDIUM(41~60) 구간에 2개 이상 패키지")
print(f"     MEDIUM 구간 패키지 수: {medium_count}개")
print(f"     결과: {'✅ 검증됨' if h5_pass else '❌ 미검증'} (기준: 2개 이상)")

passed = sum([h1_pass, h2_pass, h3_pass, h4_pass, h5_pass])
print(f"\n{'─'*70}")
print(f"가설 검증 결과: {passed}/5 통과")

## Step 15. 오탐/미탐 분석

In [ ]:
print("=" * 70)
print(f"오탐/미탐 분석 (임계값: {THRESHOLD}점)")
print("=" * 70)

df["y_true"] = [BINARY_LABEL[t] for t in df["true_label"]]
df["y_pred"] = (df["score"] >= THRESHOLD).astype(int)

# False Positives: 실제 안전이지만 위험으로 예측
fp_cases = df[(df["y_true"] == 0) & (df["y_pred"] == 1)]
print(f"\n🔴 오탐(False Positive): {len(fp_cases)}개 — 정상 패키지가 위험으로 분류됨")
for _, row in fp_cases.iterrows():
    active = [(k, v) for k, v in row["components"].items() if v > 0]
    print(f"   {row['name']:30s} 점수={row['score']:3d} | 원인: {', '.join([f'{k}(+{v})' for k,v in active])}")

# False Negatives: 실제 위험이지만 안전으로 예측
fn_cases = df[(df["y_true"] == 1) & (df["y_pred"] == 0)]
print(f"\n🟡 미탐(False Negative): {len(fn_cases)}개 — 위험 패키지가 안전으로 분류됨")
for _, row in fn_cases.iterrows():
    print(f"   {row['name']:30s} 점수={row['score']:3d} | 라벨={row['true_label']}")
    print(f"   → 개선 방안: 코드 분석, 의존성 검사 등 추가 피처 필요")

# 정탐 케이스
tp_cases = df[(df["y_true"] == 1) & (df["y_pred"] == 1)]
tn_cases = df[(df["y_true"] == 0) & (df["y_pred"] == 0)]
print(f"\n✅ 정탐(True Positive): {len(tp_cases)}개 | 진음성(True Negative): {len(tn_cases)}개")

# MEDIUM 구간 특별 분석
medium_zone = df[(df["score"] >= 41) & (df["score"] <= 60)]
print(f"\n⚠️  MEDIUM(41~60) 구간 패키지: {len(medium_zone)}개")
for _, row in medium_zone.iterrows():
    print(f"   {row['name']:30s} 점수={row['score']:3d} | → 추가 검토 필요")

## Step 16. 최종 보고서 요약

In [ ]:
print("=" * 70)
print("최종 보고서 요약")
print("=" * 70)

print("""
【프로젝트 개요】
  - 목표: npm 패키지 Slopsquatting 공격 탐지를 위한 위험도 평가 시스템
  - 방식: 규칙 기반 점수 산정 + Gemini 자연어 설명 생성 (하이브리드)
  - 데이터: npm Registry API 실시간 수집 (20개 패키지)
""")

print("【방법론 선택 이유】")
print("  - 기존 RF/XGBoost(v1/v2): 결과가 0/100으로 이진화 → 실용성 부족")
print("  - 규칙 기반 가중치: 각 위험 요소를 독립적으로 점수화 → 연속 분포")
print("  - Gemini 설명 생성: 보안 전문가 수준의 판단 근거 제공")

print("\n【점수 산정 8개 요소 (최대 100점)】")
items = [
    ("타이포스쿼팅 위험도", 30, "유명 패키지와 편집 거리 기반"),
    ("패키지 미존재(404)", 15, "npm registry에 없는 경우"),
    ("설치 스크립트", 20, "preinstall/install/postinstall"),
    ("설명 없음", 8, "description 필드 비어있음"),
    ("저장소 링크 없음", 5, "repository 필드 없음"),
    ("낮은 다운로드 수", 10, "주간 다운로드 < 10,000"),
    ("관리자 수 부족", 10, "0~2명인 경우"),
    ("키워드 없음", 2, "keywords 비어있음"),
]
for name, max_score, desc in items:
    print(f"  {name:20s}: 최대 {max_score:2d}점  ({desc})")

print(f"\n【데이터셋 분포】")
for cat in ["SAFE", "LOW", "MEDIUM", "HIGH", "CRITICAL"]:
    subset = df[df["true_label"] == cat]
    if len(subset) > 0:
        print(f"  {cat:10s}: {len(subset)}개, 평균 {subset['score'].mean():.1f}점")

print(f"\n【성능 평가】")
print(f"  F1-Score   : {f1:.3f}")
print(f"  Precision  : {precision:.3f}")
print(f"  Recall     : {recall:.3f}")
print(f"  오탐(FP)   : {fp}개")
print(f"  미탐(FN)   : {fn}개")

print("\n【한계점】")
print("  1. 코드 분석 미포함: tarball 다운로드 후 eval/obfuscation 패턴 감지 필요")
print("  2. 보안 권고 이력 미반영: CVE, npm audit 결과 연동 필요")
print("  3. 규칙 가중치 주관적: 실제 보안 사고 데이터로 가중치 최적화 필요")

print("\n【향후 개선 방향】")
print("  1. npm tarball 다운로드 + AST 분석으로 코드 패턴 감지")
print("  2. GitHub Advisory / OSV 데이터베이스 연동")
print("  3. 시계열 분석: 버전 변화 속도, maintainer 교체 이력")
print("  4. 실제 악성 패키지 데이터로 가중치 학습 (regression 모델)")

In [ ]:
# 결과를 CSV로 저장
export_cols = ["name", "true_label", "score", "grade", "exists",
               "weekly_downloads", "maintainers_count", "has_install_script",
               "min_edit_dist", "closest_package"]
df[export_cols].sort_values("score", ascending=False).to_csv("npm_risk_scores.csv", index=False)
print("💾 결과 저장 완료: npm_risk_scores.csv")
print("\n생성된 파일:")
print("  📊 risk_scores_by_package.png")
print("  📊 risk_distribution.png")
print("  📊 risk_factor_heatmap.png")
print("  📊 label_vs_score.png")
print("  📋 npm_risk_scores.csv")